# LeetCode #175 — Combine Two Tables (Ghép hai bảng dữ liệu)

**Vị trí 40 trong danh sách Easy. Bài SQL; thực hành bằng Python 3 + SQLite ngay trong Jupyter.**  
Đề gốc: https://leetcode.com/problems/combine-two-tables/

> **Phân biệt quan trọng:** Trên LeetCode, bài Database này yêu cầu **câu lệnh SQL**, không phải `class Solution` bằng Python. Trong notebook, Python `sqlite3` chỉ là công cụ dựng dữ liệu và chạy câu SQL thật. **Khi submit, chỉ dán `SELECT ... LEFT JOIN ...`**, không dán code Python.

**Mục tiêu học:**
1. Hiểu bảng (table), dòng (row), cột (column), khóa chính (primary key), khóa liên kết và `NULL`.
2. Tự viết `SELECT`, `FROM`, `WHERE`, `JOIN`, `LEFT JOIN`, `ON` và đặt bí danh.
3. Tự suy ra vì sao phải giữ **mọi người trong bảng Person**, kể cả khi không có địa chỉ.
4. Đọc bảng trace `Person × Address`, dự đoán khi nào có `NULL` và hiểu `JOIN` khác `LEFT JOIN`.
5. Nộp SQL chuẩn, test trường hợp rỗng / thiếu khớp / nhiều địa chỉ, và làm 6 bài tập có gợi ý + đáp án cuối.

**Cách học:** đọc từng mục, trước code hãy đoán kết quả truy vấn. Không cần cài database server, không dùng pandas hoặc thư viện ngoài.

## 1. Đọc đề bằng lời dễ hiểu

Công ty có **hai bảng**:

**Person** — thông tin một người:

| personId | lastName | firstName |
|---:|---|---|
| 1 | Wang | Allen |
| 2 | Alice | Bob |

**Address** — thông tin địa chỉ:

| addressId | personId | city | state |
|---:|---:|---|---|
| 1 | 2 | New York City | New York |
| 2 | 3 | Leetcode | California |

**Yêu cầu**: xuất bốn cột **`firstName, lastName, city, state`**, với **mọi dòng của Person**. Người chưa có địa chỉ phải xuất `NULL` ở `city` và `state`. Thứ tự dòng bất kỳ.

**Kết quả mẫu:** Allen Wang | NULL | NULL; Bob Alice | New York City | New York.

Đừng bỏ Allen: đây là tình huống quyết định ta phải dùng **LEFT JOIN**, chứ không phải JOIN thông thường.

## 2. SQL khác Python như thế nào?

**SQL** (Structured Query Language) là ngôn ngữ đặt câu hỏi cho cơ sở dữ liệu: “hãy lấy những cột nào?”, “từ bảng nào?”, “ghép hai bảng theo điều kiện gì?”.

Ví dụ `SELECT firstName FROM Person;` có nghĩa: “Lấy cột firstName từ bảng Person”. Dấu `;` đánh dấu kết thúc câu SQL. SQL không dùng thụt đầu dòng để quyết định cấu trúc như Python; ta xuống dòng cho dễ đọc.

`SELECT` và `FROM` là **từ khóa SQL**; `Person`, `firstName` là **tên bảng / cột** trong bài. Chúng không phải cú pháp Python. Để thực thi trong notebook, ta gửi **chuỗi SQL** cho `sqlite3`.

## 3. Python `sqlite3` — một cơ sở dữ liệu nhỏ ở bộ nhớ RAM

Để học SQL bằng notebook Python mà không phải cài MySQL server, ta dùng `sqlite3` đi kèm Python.

- `sqlite3.connect(':memory:')`: tạo database trong RAM, không ghi ra ổ đĩa.
- `conn.execute(sql)`: bảo database thực thi chuỗi SQL.
- `cursor.fetchall()`: đọc toàn bộ dòng kết quả trả về.
- `conn.close()`: đóng kết nối khi không cần.

**SQLite và MySQL khác hệ quản trị dữ liệu**, nhưng các câu `SELECT`, `JOIN`, `LEFT JOIN`, `ON`, `IS NULL` trong bài này tương thích về logic với MySQL của LeetCode. Cell SQL nộp bài vẫn sẽ là SQL thuần.

In [1]:
import sqlite3
conn = sqlite3.connect(':memory:')
print('Đã tạo database SQLite trong RAM')
print('Phiên bản SQLite:',sqlite3.sqlite_version)

Đã tạo database SQLite trong RAM
Phiên bản SQLite: 3.46.1


## 4. SQL tạo bảng: `CREATE TABLE` nghĩa là gì?

Một **bảng (table)** giống một trang tính có các cột được định nghĩa trước. **Mỗi dòng (row)** là một bản ghi.

Trong SQL:
```sql
CREATE TABLE Person (
    personId INTEGER PRIMARY KEY,
    lastName TEXT,
    firstName TEXT
);
```

- `INTEGER`: giá trị nguyên.
- `TEXT`: giá trị chữ.
- `PRIMARY KEY`: khóa chính, dùng để xác định duy nhất một dòng; ở đây là `personId`.

Đề chính thức dùng kiểu `int` và `varchar`; để chạy trong SQLite ta dùng `INTEGER` và `TEXT` tương đương cho phần thực hành. **Đây là phần dựng phòng lab**, không phải đoạn cần nộp.

In [2]:
conn.executescript("""
CREATE TABLE Person (
    personId INTEGER PRIMARY KEY,
    lastName TEXT,
    firstName TEXT
);
CREATE TABLE Address (
    addressId INTEGER PRIMARY KEY,
    personId INTEGER,
    city TEXT,
    state TEXT
);
""")
print('Đã tạo Person và Address')

Đã tạo Person và Address


## 5. `INSERT`: đưa dữ liệu mẫu vào bảng

`INSERT INTO Person VALUES (...)` là thêm **một dòng dữ liệu**. Ở đây ta dùng `executemany` của Python để thêm nhiều dòng; ký hiệu `?` là vị trí của một giá trị sẽ được Python truyền vào an toàn.

Điểm cần quan sát: Address có dòng `personId = 3` trong khi Person không có người số 3. Đây là địa chỉ **không khớp người** theo dữ liệu mẫu; nó **không tạo ra** người thứ ba trong kết quả. Ngược lại, Person 1 không có địa chỉ, nhưng **vẫn phải xuất hiện**.

In [3]:
conn.executemany('INSERT INTO Person VALUES (?,?,?)', [
    (1,'Wang','Allen'), (2,'Alice','Bob')
])
conn.executemany('INSERT INTO Address VALUES (?,?,?,?)', [
    (1,2,'New York City','New York'),(2,3,'Leetcode','California')
])
conn.commit()
print('Dữ liệu đã được chèn thành công')

Dữ liệu đã được chèn thành công


## 6. Hàm hiển thị kết quả SQL — chỉ phục vụ việc học

Ta viết một hàm nhỏ `show_sql(query, db=conn)` để **chạy truy vấn và in ra cột/dòng**. Mọi câu SQL tiếp theo vẫn là SQL thực sự: Python chỉ đóng vai trò truyền chuỗi và hiển thị.

`cursor.description` lưu tên các cột kết quả; `cursor.fetchall()` lấy danh sách các dòng. Python hiển thị `None` khi SQL trả giá trị `NULL`; **`None` không phải chuỗi `'None'`** và không phải số 0.

Hãy chạy thử truy vấn ngắn nhất để chắc chắn đã hiểu cách sử dụng phòng lab.

In [4]:
def show_sql(query, db=conn):
    cursor=db.execute(query)
    columns=[item[0] for item in cursor.description]
    rows=cursor.fetchall()
    print('Cột:', columns)
    for row in rows:
        print('  ',row)
    print('Số dòng:',len(rows))
    return rows

show_sql('SELECT firstName, lastName FROM Person ORDER BY personId;')

Cột: ['firstName', 'lastName']
   ('Allen', 'Wang')
   ('Bob', 'Alice')
Số dòng: 2


[('Allen', 'Wang'), ('Bob', 'Alice')]

## 7. Tư duy `SELECT` và `FROM`: ta chọn gì, từ đâu?

Câu SQL:
```sql
SELECT firstName, lastName
FROM Person;
```

Nghĩa là:
- `FROM Person`: bắt đầu xem các dòng của bảng Person.
- `SELECT firstName, lastName`: chỉ hiển thị hai cột này theo **đúng thứ tự đã ghi**.

`SELECT *` dùng để xem **mọi cột** khi khám phá. Nhưng khi nộp #175 phải chọn **đúng 4 cột**, không dùng `*` vì sẽ thừa personId/addressId.

`ORDER BY personId` chỉ thêm ở phòng lab để nhìn thứ tự cố định; đề nói trả kết quả **bất kỳ thứ tự nào**, nên câu submit không cần `ORDER BY`.

In [5]:
show_sql('SELECT * FROM Person ORDER BY personId;')
show_sql('SELECT firstName,lastName FROM Person ORDER BY personId;')

Cột: ['personId', 'lastName', 'firstName']
   (1, 'Wang', 'Allen')
   (2, 'Alice', 'Bob')
Số dòng: 2
Cột: ['firstName', 'lastName']
   ('Allen', 'Wang')
   ('Bob', 'Alice')
Số dòng: 2


[('Allen', 'Wang'), ('Bob', 'Alice')]

## 8. `WHERE` là bộ lọc từng dòng như thế nào?

Nếu muốn chỉ lấy người có `personId = 2`:
```sql
SELECT firstName FROM Person WHERE personId = 2;
```

Trong SQL, so sánh bằng nhau dùng **một dấu `=`**, không phải `==` như Python thường viết. `WHERE` giữ lại những dòng thỏa điều kiện.

Ví dụ này giúp bạn phân biệt **chọn cột** (`SELECT`) và **chọn dòng** (`WHERE`). Ta chưa ghép bảng: mọi thao tác vẫn chỉ xảy ra trên Person.

In [6]:
show_sql('SELECT personId, firstName FROM Person WHERE personId = 2;')
show_sql('SELECT personId, firstName FROM Person WHERE personId > 2;')

Cột: ['personId', 'firstName']
   (2, 'Bob')
Số dòng: 1
Cột: ['personId', 'firstName']
Số dòng: 0


[]

## 9. Hai bảng liên hệ với nhau bằng cột `personId`

Trong Address, `personId` cho biết **địa chỉ thuộc người nào**. Để ghép đúng ta phải so sánh:

`Person.personId = Address.personId`.

**Chú ý hai ID khác vai trò:**
- `Address.addressId` nhận diện dòng **địa chỉ**.
- `Address.personId` cho biết người **sở hữu** địa chỉ.

Nếu ghép `Person.personId = Address.addressId`, trong mẫu sẽ tình cờ khớp một số dòng nhưng **sai quan hệ**. Trong dữ liệu thực tế, hai loại ID độc lập nhau và hoàn toàn không có lý do phải bằng nhau.

In [7]:
show_sql('SELECT personId, city, state FROM Address ORDER BY addressId;')

Cột: ['personId', 'city', 'state']
   (2, 'New York City', 'New York')
   (3, 'Leetcode', 'California')
Số dòng: 2


[(2, 'New York City', 'New York'), (3, 'Leetcode', 'California')]

## 10. Brute Force bằng Python: duyệt từng người, tìm địa chỉ

Để hiểu phép JOIN trước khi học cú pháp, thử dùng Python thuần:

- Với mỗi dòng Person, duyệt các dòng Address.
- Nếu cùng `personId`, ghép `firstName, lastName, city, state`.
- Nếu duyệt hết Address mà **không có dòng khớp**, ghép `None, None` cho hai cột địa chỉ.

Trong phòng lab ta viết đoạn này để thấy công việc mà database có thể làm. Với `p` người và `a` địa chỉ, cách đơn giản kiểm tra đến `p×a` cặp, tức **O(p·a)** thời gian. Đây **không phải** code nộp LeetCode #175.

In [8]:
people=conn.execute('SELECT personId,lastName,firstName FROM Person').fetchall()
addresses=conn.execute('SELECT addressId,personId,city,state FROM Address').fetchall()
for person_id, last_name, first_name in people:
    matches=[a for a in addresses if a[1]==person_id]
    if matches:
        for addr in matches:
            print(first_name,last_name,addr[2],addr[3])
    else:
        print(first_name,last_name,None,None)

Allen Wang None None
Bob Alice New York City New York


## 11. Bảng trace ghép theo khóa — trước khi học `JOIN`

Đặt hai cột `Person.personId` và `Address.personId` cạnh nhau:

| Người đang xét | ID người | Dòng địa chỉ khớp | Kết luận |
|---|---:|---|---|
| Allen Wang | 1 | Không có | **Giữ Allen**, thêm `NULL`, `NULL` |
| Bob Alice | 2 | Address ID 1 | Ghép New York City, New York |

Còn Address có `personId=3`? Không có Person ID 3 nên **bỏ qua** khi xuất báo cáo bắt đầu từ Person.

Đây là quy tắc “**giữ tất cả dòng bên trái**” (left side). Chữ **LEFT** trong `LEFT JOIN` chính là “bảng đặt ở bên trái cú pháp JOIN”, ở đây phải là **Person**.

## 12. SQL `INNER JOIN`: chỉ giữ các cặp có khóa khớp

Trước tiên, thử `INNER JOIN`:
```sql
FROM Person AS p
INNER JOIN Address AS a
    ON p.personId = a.personId
```

`AS p` và `AS a` là **bí danh**: viết gọn khi hai bảng đều có cột tên `personId`. `p.personId` nghĩa là cột personId **của bảng p**.

`ON` chỉ ra điều kiện **hai dòng được ghép**. Với `INNER JOIN`, dòng nào không có đối tác sẽ bị bỏ, vì thế Allen biến mất. Đây là **phản ví dụ** cho thấy `INNER JOIN` không đúng yêu cầu #175.

In [9]:
SQL_INNER = """
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person AS p
INNER JOIN Address AS a
    ON p.personId = a.personId;
"""
show_sql(SQL_INNER)

Cột: ['firstName', 'lastName', 'city', 'state']
   ('Bob', 'Alice', 'New York City', 'New York')
Số dòng: 1


[('Bob', 'Alice', 'New York City', 'New York')]

## 13. `LEFT JOIN`: giữ mọi dòng ở bảng bên trái

`LEFT JOIN` (cũng viết `LEFT OUTER JOIN`) hoạt động như sau:

1. Lấy từng dòng bảng **bên trái** (`Person`).
2. Nếu bảng bên phải (`Address`) có dòng khớp điều kiện `ON`, ghép các cột lại.
3. Nếu **không có** dòng khớp, vẫn giữ dòng Person và lấp các cột của Address bằng **SQL NULL**.

Với dữ liệu ví dụ: kết quả có **Allen và Bob**. Đây chính là yêu cầu đề bài.

**Mẹo đọc:** đọc `FROM Person LEFT JOIN Address` thành “từ Person, nối thêm địa chỉ nếu tìm được; không tìm được vẫn giữ người”.

In [10]:
SQL_LEFT = """
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person AS p
LEFT JOIN Address AS a
    ON p.personId = a.personId;
"""
show_sql(SQL_LEFT)

Cột: ['firstName', 'lastName', 'city', 'state']
   ('Allen', 'Wang', None, None)
   ('Bob', 'Alice', 'New York City', 'New York')
Số dòng: 2


[('Allen', 'Wang', None, None), ('Bob', 'Alice', 'New York City', 'New York')]

## 14. So sánh `INNER` và `LEFT`: đếm số dòng để thấy khác biệt

Trên dữ liệu mẫu:

| Loại ghép | Allen có xuất hiện? | Bob có xuất hiện? | Số dòng |
|---|---|---|---:|
| INNER JOIN | Không | Có | 1 |
| LEFT JOIN | **Có (NULL địa chỉ)** | Có | **2** |

Không phải JOIN nào cũng giữ bằng số dòng bảng trái. **Một người có nhiều dòng địa chỉ** có thể tạo nhiều dòng kết quả. Điều được `LEFT JOIN` bảo đảm là: mỗi dòng bên trái có **ít nhất một dòng kết quả** (trong trường hợp quan hệ ghép 1–n), không bảo đảm **đúng một dòng**.

Đề nêu khóa chính `Address.addressId`, **không khẳng định `Address.personId` là duy nhất**. Vì vậy không nên tự thêm `DISTINCT` để xóa bớt dữ liệu.

In [11]:
print('Số người:',conn.execute('SELECT COUNT(*) FROM Person').fetchone()[0])
print('INNER JOIN:',len(conn.execute(SQL_INNER).fetchall()))
print('LEFT JOIN:',len(conn.execute(SQL_LEFT).fetchall()))
assert len(conn.execute(SQL_INNER).fetchall())==1
assert len(conn.execute(SQL_LEFT).fetchall())==2

Số người: 2
INNER JOIN: 1
LEFT JOIN: 2


## 15. `NULL` là gì? Nó không phải `0` hay chuỗi rỗng

**SQL `NULL`** biểu thị giá trị vắng mặt/không xác định. Phòng lab Python hiển thị thành `None`. Cần phân biệt:

- `NULL`: không có giá trị.
- `0`: có giá trị số 0.
- `''`: có chuỗi, nhưng chuỗi rỗng.
- `'NULL'`: có **bốn chữ** N-U-L-L, không phải NULL.

`LEFT JOIN` tự cung cấp `NULL` cho những cột thuộc bảng phải khi không có dòng khớp. Ta **không cần** tự viết `NULL AS city` trong lời giải chính; làm vậy sẽ khiến Bob cũng có city NULL dù đã có địa chỉ.

In [12]:
show_sql("SELECT NULL AS missing, 0 AS zero_value, '' AS empty_text, 'NULL' AS literal_text;")

Cột: ['missing', 'zero_value', 'empty_text', 'literal_text']
   (None, 0, '', 'NULL')
Số dòng: 1


[(None, 0, '', 'NULL')]

## 16. `IS NULL` thay vì `= NULL` — lỗi SQL rất phổ biến

SQL dùng logic có giá trị **UNKNOWN** (không xác định) khi so sánh thông thường với `NULL`.

- `city = NULL` **không** cho True kể cả khi city đang NULL.
- `city IS NULL` mới là cách kiểm tra SQL NULL.
- `city IS NOT NULL` kiểm tra có giá trị.

Ta thử với kết quả LEFT JOIN: `WHERE a.personId IS NULL` lấy các dòng Person **không có bất kỳ dòng Address khớp nào**. Nếu dùng `a.city IS NULL`, còn có thể bắt trúng người **có dòng địa chỉ nhưng city chưa được ghi**; do đó kiểm tra cột khóa khớp (`a.personId`) thường chính xác hơn để tìm thiếu bản ghi.

In [13]:
show_sql("""SELECT p.firstName, a.personId, a.city
FROM Person AS p LEFT JOIN Address AS a ON p.personId=a.personId
WHERE a.personId IS NULL;""")
print('NULL = NULL ->',conn.execute('SELECT NULL = NULL').fetchone()[0])

Cột: ['firstName', 'personId', 'city']
   ('Allen', None, None)
Số dòng: 1
NULL = NULL -> None


## 17. Từng phần của truy vấn cuối được đọc như thế nào?

Đọc theo logic:

```sql
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person AS p
LEFT JOIN Address AS a
    ON p.personId = a.personId;
```

1. **FROM Person p** — tập người cần báo cáo.
2. **LEFT JOIN Address a ON ...** — tìm địa chỉ theo *người ID*, giữ người khi không có khớp.
3. **SELECT ...** — trình bày đúng 4 cột và đúng thứ tự.

Thứ tự viết SQL bắt đầu từ `SELECT`, nhưng khi suy nghĩ ta thường hình dung **FROM/JOIN trước, SELECT sau**. Mỗi mệnh đề phục vụ một yêu cầu cụ thể, không có dòng code thừa.

In [14]:
SQL_SUBMIT_175 = """
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person AS p
LEFT JOIN Address AS a
    ON p.personId = a.personId;
"""
expected_175={('Allen','Wang',None,None),('Bob','Alice','New York City','New York')}
assert set(conn.execute(SQL_SUBMIT_175).fetchall())==expected_175
print('Câu SQL chuẩn đã qua test mẫu: PASS')

Câu SQL chuẩn đã qua test mẫu: PASS


## 18. Mô phỏng từng cặp bằng SQL: nhìn khóa trước rồi nhìn kết quả

Một cách học tốt là **hiển thị cả hai cột khóa** trong **bảng trace**, dù khi nộp không xuất chúng.

```sql
SELECT p.personId AS pid,
       a.personId AS matched_pid,
       p.firstName, a.city
...
```

- Allen có `pid = 1`, `matched_pid = NULL` → không có dòng Address.
- Bob có `pid = 2`, `matched_pid = 2` → ghép đúng.

`AS pid` đặt tên **cột kết quả**, không làm thay đổi dữ liệu bảng. Đây là kỹ thuật debug SQL tương đương `print` giá trị biến trong Python.

In [15]:
TRACE_SQL_175="""
SELECT p.personId AS pid, a.personId AS matched_pid,
       p.firstName, a.city
FROM Person AS p
LEFT JOIN Address AS a ON p.personId=a.personId
ORDER BY p.personId;
"""
show_sql(TRACE_SQL_175)

Cột: ['pid', 'matched_pid', 'firstName', 'city']
   (1, None, 'Allen', None)
   (2, 2, 'Bob', 'New York City')
Số dòng: 2


[(1, None, 'Allen', None), (2, 2, 'Bob', 'New York City')]

## 19. Vì sao `ON` phải ghép hai cột `personId`?

Hãy so sánh hai điều kiện:

- **Đúng:** `p.personId = a.personId` — hai dòng mô tả **cùng một người**.
- **Sai:** `p.personId = a.addressId` — ID người trùng ID dòng địa chỉ chỉ là **sự tình cờ**.

Với mẫu đề, nếu ghép sai thì Allen bị nối với dòng địa chỉ có addressId 1 (thực ra thuộc Bob). Đó là lỗi dữ liệu nghiêm trọng dù cú pháp SQL vẫn hợp lệ.

**Bài học tổng quát:** `ON` phải biểu diễn **quan hệ trong mô hình dữ liệu**, không ghép các số có vẻ giống nhau.

In [16]:
WRONG_JOIN="""
SELECT p.firstName, a.city
FROM Person AS p LEFT JOIN Address AS a
ON p.personId = a.addressId ORDER BY p.personId;
"""
show_sql(WRONG_JOIN)
print('Đối chiếu kết quả đúng:')
show_sql('SELECT p.firstName,a.city FROM Person p LEFT JOIN Address a ON p.personId=a.personId ORDER BY p.personId')

Cột: ['firstName', 'city']
   ('Allen', 'New York City')
   ('Bob', 'Leetcode')
Số dòng: 2
Đối chiếu kết quả đúng:
Cột: ['firstName', 'city']
   ('Allen', None)
   ('Bob', 'New York City')
Số dòng: 2


[('Allen', None), ('Bob', 'New York City')]

## 20. Vì sao không nên dùng `WHERE a.city = ...` vô ý trong LEFT JOIN?

Một bẫy thực tế: sau `LEFT JOIN`, người không có địa chỉ nhận `a.city = NULL`. Nếu viết:
```sql
WHERE a.city = 'New York City'
```
thì Allen bị **lọc mất**, trái với mục tiêu giữ tất cả người.

Nếu bạn thật sự muốn **chỉ lấy người ở New York City**, dùng WHERE là đúng. Nhưng nếu bạn muốn **giữ mọi người** và chỉ ghép địa chỉ phù hợp thành phố đó thì điều kiện hạn chế địa chỉ phải đặt trong **`ON`**.

Đây là hai bài toán **khác nhau**, không phải ON và WHERE có thể hoán đổi tùy ý.

In [17]:
FILTER_WHERE="""SELECT p.firstName,a.city FROM Person p LEFT JOIN Address a
ON p.personId=a.personId WHERE a.city='New York City';"""
FILTER_ON="""SELECT p.firstName,a.city FROM Person p LEFT JOIN Address a
ON p.personId=a.personId AND a.city='New York City';"""
print('Lọc trong WHERE:')
show_sql(FILTER_WHERE)
print('Giới hạn khớp trong ON:')
show_sql(FILTER_ON)

Lọc trong WHERE:
Cột: ['firstName', 'city']
   ('Bob', 'New York City')
Số dòng: 1
Giới hạn khớp trong ON:
Cột: ['firstName', 'city']
   ('Allen', None)
   ('Bob', 'New York City')
Số dòng: 2


[('Allen', None), ('Bob', 'New York City')]

## 21. Một người có hai địa chỉ thì `LEFT JOIN` cho mấy dòng?

**Đề chỉ xác nhận khóa chính của Address là `addressId`.** Hai dòng Address có `addressId` khác nhau vẫn có thể cùng một `personId` nếu không có ràng buộc duy nhất bổ sung.

Giả sử Bob có hai địa chỉ ở hai thành phố. `LEFT JOIN` trả về **hai dòng của Bob**, vì tìm thấy **hai cặp khớp**. Allen vẫn trả một dòng với NULL. Đây là hành vi đúng của phép JOIN.

Khi cần “mỗi người đúng một địa chỉ”, phải có **quy tắc chọn địa chỉ nào**. Bài #175 **không yêu cầu** chọn một địa chỉ nên không tự ý thêm logic chọn, nhóm hay xóa bản ghi.

In [18]:
lab_multi=sqlite3.connect(':memory:')
conn.backup(lab_multi)
lab_multi.execute("INSERT INTO Address VALUES (3,2,'Boston','Massachusetts')")
show_sql(SQL_SUBMIT_175+' ',lab_multi)
assert len(lab_multi.execute(SQL_SUBMIT_175).fetchall())==3
lab_multi.close()

Cột: ['firstName', 'lastName', 'city', 'state']
   ('Allen', 'Wang', None, None)
   ('Bob', 'Alice', 'Boston', 'Massachusetts')
   ('Bob', 'Alice', 'New York City', 'New York')
Số dòng: 3


## 22. Trường hợp bảng Address rỗng vẫn phải có mọi Person

Nếu **không có địa chỉ nào**, tất cả người trong bảng Person đều phải xuất hiện với `city` và `state` là NULL. Dùng `INNER JOIN` lúc đó sẽ trả về **bảng rỗng** — sai.

Ta tạo một cơ sở dữ liệu phòng lab riêng (không phá dữ liệu mẫu gốc) bằng `conn.backup(...)`, sau đó xóa các dòng Address để kiểm tra chính xác nhánh này.

In [19]:
lab_empty=sqlite3.connect(':memory:')
conn.backup(lab_empty)
lab_empty.execute('DELETE FROM Address')
rows=show_sql(SQL_SUBMIT_175,lab_empty)
assert len(rows)==2 and all(city is None and state is None for _,_,city,state in rows)
lab_empty.close()
print('Test Address rỗng: PASS')

Cột: ['firstName', 'lastName', 'city', 'state']
   ('Allen', 'Wang', None, None)
   ('Bob', 'Alice', None, None)
Số dòng: 2
Test Address rỗng: PASS


## 23. Trường hợp bảng Person rỗng: kết quả ra sao?

`LEFT JOIN` đảm bảo **giữ các dòng đang có ở bảng bên trái**. Nếu Person không có dòng nào thì **không có người để xuất**, dù Address có bao nhiêu dòng.

Chú ý: `LEFT JOIN` không có nghĩa là “luôn trả ít nhất một dòng”; nó có nghĩa là **không xóa dòng bảng trái chỉ vì thiếu dòng bảng phải**. Khi bảng trái rỗng, kết quả rỗng là hợp lý.

In [20]:
lab_no_people=sqlite3.connect(':memory:')
conn.backup(lab_no_people)
lab_no_people.execute('DELETE FROM Person')
assert lab_no_people.execute(SQL_SUBMIT_175).fetchall()==[]
print('Test Person rỗng: PASS')
lab_no_people.close()

Test Person rỗng: PASS


## 24. Nếu `city` thật sự NULL trong bản ghi Address?

Có **hai lý do khác nhau** để nhìn thấy `city` NULL:

1. Người không có bản ghi Address khớp → LEFT JOIN tạo NULL cho **cả dòng bên phải**.
2. Người có bản ghi Address nhưng **cột `city` trong bản ghi vốn NULL**.

Để phân biệt, hiển thị `a.addressId` hoặc `a.personId` cùng city. Khi không có khớp, các cột phía Address đều NULL; khi có bản ghi nhưng city thiếu, `addressId` vẫn có giá trị.

Vì vậy khi tìm “người chưa có địa chỉ” nên kiểm tra **khóa bản ghi phải**, không dùng riêng `a.city IS NULL`.

In [21]:
lab_null_city=sqlite3.connect(':memory:')
conn.backup(lab_null_city)
lab_null_city.execute('UPDATE Address SET city=NULL WHERE addressId=1')
show_sql("""SELECT p.firstName,a.addressId,a.city
FROM Person p LEFT JOIN Address a ON p.personId=a.personId
ORDER BY p.personId;""",lab_null_city)
lab_null_city.close()

Cột: ['firstName', 'addressId', 'city']
   ('Allen', None, None)
   ('Bob', 1, None)
Số dòng: 2


## 25. Biến thể 1: chỉ tìm những người CHƯA có địa chỉ

Nếu câu hỏi đổi thành “ai chưa có địa chỉ?”, ta vẫn dùng LEFT JOIN nhưng thêm:

```sql
WHERE a.personId IS NULL
```

Tại sao? Dòng không khớp có các cột bên Address bằng NULL. Do đó chỉ giữ các dòng đó. Kỹ thuật này thường gọi là **anti-join** (lọc những dòng bên trái không có đối tác bên phải).

Đây là bài biến thể, **không phải lời giải nộp #175**; nó hữu ích để nhận diện câu SQL “có nhưng không có” ở các bài tiếp theo.

In [22]:
SQL_MISSING_175="""
SELECT p.firstName, p.lastName
FROM Person AS p
LEFT JOIN Address AS a ON p.personId = a.personId
WHERE a.personId IS NULL;
"""
assert conn.execute(SQL_MISSING_175).fetchall()==[('Allen','Wang')]
show_sql(SQL_MISSING_175)

Cột: ['firstName', 'lastName']
   ('Allen', 'Wang')
Số dòng: 1


[('Allen', 'Wang')]

## 26. Biến thể 2: `COALESCE` dùng khi không muốn hiển thị NULL

Trong báo cáo thật, ta đôi khi muốn hiển thị chữ “Chưa có địa chỉ”. SQL có hàm:

`COALESCE(a.city, 'Chưa có địa chỉ')`

Nó trả giá trị đầu tiên **không NULL**. Nếu city có giá trị thì giữ nguyên; nếu NULL thì dùng chữ dự phòng.

**Tuy nhiên**: đề #175 yêu cầu **NULL** khi thiếu địa chỉ. Vì vậy **không dùng COALESCE trong câu submit**, dù biến thể này hữu ích ngoài đời.

In [23]:
show_sql("""
SELECT p.firstName,COALESCE(a.city,'Chưa có địa chỉ') AS city
FROM Person AS p LEFT JOIN Address AS a ON p.personId=a.personId
ORDER BY p.personId;
""")

Cột: ['firstName', 'city']
   ('Allen', 'Chưa có địa chỉ')
   ('Bob', 'New York City')
Số dòng: 2


[('Allen', 'Chưa có địa chỉ'), ('Bob', 'New York City')]

## 27. Vì sao truy vấn đúng? Chứng minh bằng cách ghép từng dòng

Đây là lời giải thích tính đúng bằng trực giác:

1. Lấy một dòng `p` bất kỳ trong Person. Lệnh `FROM Person AS p` đảm bảo ta bắt đầu từ dòng đó.
2. `LEFT JOIN Address AS a ON p.personId = a.personId` xét **đúng những địa chỉ mang cùng personId**; vì vậy không gán địa chỉ người khác.
3. Nếu có địa chỉ khớp, kết quả chứa các dòng đã ghép. Nếu không có, **LEFT JOIN vẫn xuất p** và đưa NULL cho các cột địa chỉ.
4. `SELECT` lấy bốn cột được yêu cầu, không thay đổi quan hệ vừa xác định.

Vậy mọi người đều được báo cáo, các địa chỉ (nếu có) thuộc đúng người, và người thiếu địa chỉ được điền NULL: **đúng đề**.

## 28. Phân tích độ phức tạp SQL: phải nói rõ giả định

Với truy vấn SQL, **không có một Big-O tuyệt đối duy nhất** nếu chưa biết bộ máy database sẽ chọn kế hoạch thực thi nào.

Ký hiệu: `p` = số dòng Person, `a` = số dòng Address, `k` = số dòng kết quả.

- **Python baseline** quét toàn bộ Address cho từng người: O(p·a), cần O(k) để lưu kết quả nếu gom list.
- **Database JOIN** có thể dùng **hash join** với thời gian kỳ vọng khoảng O(p+a+k) và bộ nhớ O(a) cho bảng băm, hoặc dùng **index** để tra cứu theo personId (chi phí phụ thuộc cấu trúc và số khớp), hay nested loop O(p·a) khi không có chỉ mục / thuật toán phù hợp.
- Tối thiểu phải tốn O(k) công xuất kết quả; `k` có thể lớn hơn `p` nếu một người có nhiều địa chỉ.

**Không nên ghi “LEFT JOIN luôn O(1) bộ nhớ và O(n) thời gian”.** Độ phức tạp phụ thuộc dữ liệu, chỉ mục và query planner. Với bài nhập môn, điều quan trọng nhất là **chọn đúng phép ghép**.

## 29. Lỗi người mới: sai → nguyên nhân → sửa

| Viết sai / hiểu sai | Sai ở đâu? | Cách sửa |
|---|---|---|
| `INNER JOIN Address` | Loại người chưa có địa chỉ | Dùng `LEFT JOIN` |
| `FROM Address LEFT JOIN Person` | Giữ tất cả **địa chỉ**, không giữ mọi người | Đặt **Person bên trái** |
| `ON p.personId = a.addressId` | Ghép hai ID không cùng ý nghĩa | `p.personId = a.personId` |
| `SELECT *` | Trả thừa cột | Chọn đúng 4 cột |
| `WHERE a.city IS NOT NULL` | Vô ý xóa người thiếu địa chỉ | Bỏ bộ lọc này |
| `a.city = NULL` | So sánh NULL không hoạt động như mong muốn | `a.city IS NULL` |
| `DISTINCT` không có yêu cầu | Có thể làm mất dòng hợp lệ | Không tự thêm |

Kiểm thử tốt nhất là **trường hợp người không có địa chỉ** và **trường hợp ID địa chỉ trùng ID người khác**.

## 30. Checklist nộp SQL vào LeetCode

Bài #175 là bài **Database**. Trên trang LeetCode, chọn ngôn ngữ **MySQL** (hoặc hệ SQL được hỗ trợ), dán **một câu SELECT**:

```sql
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person AS p
LEFT JOIN Address AS a
    ON p.personId = a.personId;
```

**Không dán** `import sqlite3`, `conn.execute`, `CREATE TABLE`, `INSERT`, phần in debug. LeetCode đã dựng hai bảng dữ liệu và chỉ chờ câu truy vấn của bạn. Phòng lab SQLite có thể tự kiểm điều kiện và cột kết quả trước khi nộp.

In [24]:
cursor=conn.execute(SQL_SUBMIT_175)
actual_cols=[x[0] for x in cursor.description]
actual_rows=set(cursor.fetchall())
assert actual_cols==['firstName','lastName','city','state']
assert actual_rows==expected_175
print('Đúng tên cột, đúng tập kết quả, có NULL: PASS')

Đúng tên cột, đúng tập kết quả, có NULL: PASS


# 31. Bài tập tự luyện — 6 cấp độ, không xem đáp án trước

**Mục tiêu:** tự suy ra JOIN dựa trên yêu cầu giữ/bỏ những dòng nào. **Các cell TODO chỉ khai báo chuỗi truy vấn, không tự thực thi.** Khi điền xong, tự bỏ dấu `#` để chạy test.

1. **Dự đoán output:** Trên dữ liệu mẫu, `SELECT firstName FROM Person WHERE personId=2` trả mấy dòng, chữ gì?
2. **Trace bằng tay:** nếu Person 1 không có Address, Person 2 có Address, sau LEFT JOIN có những cặp `(personId, city)` nào?
3. **Điền TODO:** truy vấn `LEFT JOIN` đúng bốn cột.
4. **Viết từ đầu:** truy vấn `INNER JOIN` chỉ lấy người có địa chỉ, xuất `firstName, city`.
5. **Biến thể:** tìm những người chưa có **bất kỳ bản ghi Address** nào (đúng cột khóa kiểm NULL).
6. **Mini challenge:** thống kê `personId`, `firstName`, **số bản ghi Address** của mỗi người (kể cả 0); nghĩ tới `COUNT` và `GROUP BY`.

**Tự bật test** sau khi làm xong. Đáp án mẫu và giải thích đủ ở phần cuối.

In [25]:
# Bài 1,2: tự tính bằng tay, chưa chạy câu SQL kiểm tra.
# Ghi vào giấy: (a) số dòng SELECT WHERE personId=2; (b) cặp sau LEFT JOIN.
practice_person_ids=[1,2]

### Bài 3 — Điền chỗ trống trong câu SQL

Chọn đúng bốn cột; dùng **Person bên trái** và ghép `personId` của hai bảng.

```sql
SELECT ...
FROM ... AS p
LEFT JOIN ... AS a ON ...;
```

Không dùng `SELECT *` vì sai yêu cầu định dạng output.

In [26]:
EX_175_3 = """
-- TODO: viết SELECT 4 cột, FROM Person, LEFT JOIN Address, ON khóa
"""
# Khi làm xong mới bỏ #:
# assert set(conn.execute(EX_175_3).fetchall()) == expected_175

### Bài 4 — Viết INNER JOIN độc lập

Lấy `firstName, city` của những người **có địa chỉ**. Vì không cần giữ người thiếu địa chỉ, `INNER JOIN` là phù hợp. Hãy tự viết mà không nhìn đoạn mẫu.

In [27]:
EX_175_4 = """
-- TODO: SELECT firstName, city từ INNER JOIN
"""
# assert conn.execute(EX_175_4).fetchall() == [('Bob','New York City')]

### Bài 5 — Tìm người chưa có địa chỉ

Dùng LEFT JOIN trước; khi không có dòng Address khớp, cột `a.personId` là NULL. Dùng `WHERE a.personId IS NULL` để chỉ giữ những người này.

In [28]:
EX_175_5 = """
-- TODO: LEFT JOIN rồi WHERE ... IS NULL
"""
# assert conn.execute(EX_175_5).fetchall() == [('Allen',)]

### Bài 6 — Mini challenge: đếm địa chỉ theo người

Nếu dùng `COUNT(*)`, một người **không có địa chỉ** cũng có **một dòng** do LEFT JOIN tạo ra, dẫn đến đếm sai là 1.

Hãy dùng `COUNT(a.addressId)` vì `COUNT(cột)` **bỏ qua NULL**, kết hợp `GROUP BY p.personId, p.firstName`. Đề #175 không yêu cầu việc thống kê này; đây là biến thể để rèn cách suy luận NULL.

In [29]:
EX_175_6 = """
-- TODO: SELECT id, firstName, COUNT(a.addressId) AS addressCount
-- TODO: LEFT JOIN ... GROUP BY ...
"""
# expected_count=[(1,'Allen',0),(2,'Bob',1)]
# assert conn.execute(EX_175_6).fetchall() == expected_count

## 32. Gợi ý 3 mức cho 6 bài tập

| Bài | Mức 1: nhớ mục tiêu | Mức 2: nhớ cú pháp | Mức 3: gần đáp án |
|---|---|---|---|
| 1 | Chỉ người có ID=2 | `WHERE personId=2` | Bob, một dòng |
| 2 | Giữ **mọi** Person | không có khớp → NULL | Allen→NULL; Bob→New York City |
| 3 | Bốn cột đúng thứ tự | `FROM Person p LEFT JOIN Address a` | `ON p.personId=a.personId` |
| 4 | Chỉ giữ cặp khớp | `JOIN ... ON` | `SELECT p.firstName,a.city` |
| 5 | Bản ghi phải không tồn tại | `IS NULL` | `WHERE a.personId IS NULL` |
| 6 | Đếm giá trị **không NULL** | `COUNT(a.addressId)` | `GROUP BY p.personId,p.firstName` |

Nếu bí, chỉ đọc từng mức. Đáp án giải thích có ở các mục kế tiếp.

## 33. Đáp án Bài 1 và Bài 2: dự đoán, trace

**Bài 1:** `Bob`; **1 dòng**. Vì `WHERE personId=2` chỉ giữ người có ID 2.

**Bài 2:** `(1, NULL)` của Allen và `(2, 'New York City')` của Bob. Lưu ý `Address.personId=3` không có Person cùng ID nên **không xuất hiện**.

Đây là câu hỏi đọc hiểu, không cần ghi nhớ syntax dài.

In [30]:
assert conn.execute('SELECT firstName FROM Person WHERE personId=2').fetchall()==[('Bob',)]
rows=conn.execute("""SELECT p.personId,a.city
FROM Person p LEFT JOIN Address a ON p.personId=a.personId
ORDER BY p.personId""").fetchall()
assert rows==[(1,None),(2,'New York City')]
print('Đáp án Bài 1–2: PASS')

Đáp án Bài 1–2: PASS


## 34. Đáp án Bài 3 — LEFT JOIN giữ mọi người

Cấu trúc giải đúng: xác định bảng bắt buộc giữ là Person, đặt ở **bên trái**, dùng `ON` theo đúng khóa quan hệ; `SELECT` chỉ lấy bốn cột.

In [31]:
ANSWER_175_3="""
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person AS p
LEFT JOIN Address AS a ON p.personId=a.personId;
"""
assert set(conn.execute(ANSWER_175_3).fetchall())==expected_175
show_sql(ANSWER_175_3)

Cột: ['firstName', 'lastName', 'city', 'state']
   ('Allen', 'Wang', None, None)
   ('Bob', 'Alice', 'New York City', 'New York')
Số dòng: 2


[('Allen', 'Wang', None, None), ('Bob', 'Alice', 'New York City', 'New York')]

## 35. Đáp án Bài 4 — chỉ lấy người có địa chỉ

`INNER JOIN` bỏ Allen vì không có bản ghi Address nào khớp. Đây là trường hợp **INNER JOIN đúng với yêu cầu biến thể**, nhưng **sai với #175 gốc**.

In [32]:
ANSWER_175_4="""
SELECT p.firstName, a.city
FROM Person AS p
INNER JOIN Address AS a ON p.personId=a.personId;
"""
assert conn.execute(ANSWER_175_4).fetchall()==[('Bob','New York City')]
show_sql(ANSWER_175_4)

Cột: ['firstName', 'city']
   ('Bob', 'New York City')
Số dòng: 1


[('Bob', 'New York City')]

## 36. Đáp án Bài 5 — tìm người không có bản ghi Address

Với LEFT JOIN, khi không có khớp ta nhận các cột a.* bằng NULL. `a.personId IS NULL` sẽ lọc ra đúng những người chưa có địa chỉ; không bị nhầm với người có bản ghi địa chỉ nhưng `city` thiếu.

In [33]:
ANSWER_175_5="""
SELECT p.firstName
FROM Person AS p
LEFT JOIN Address AS a ON p.personId=a.personId
WHERE a.personId IS NULL;
"""
assert conn.execute(ANSWER_175_5).fetchall()==[('Allen',)]
show_sql(ANSWER_175_5)

Cột: ['firstName']
   ('Allen',)
Số dòng: 1


[('Allen',)]

## 37. Đáp án Bài 6 — COUNT cột thay vì COUNT(*)

`GROUP BY` gom những dòng sau JOIN theo người. `COUNT(a.addressId)` đếm số dòng bên phải **có ID thật**, không đếm dòng NULL giả do LEFT JOIN tạo ra. Nếu Bob có 2 địa chỉ, giá trị của Bob tăng thành 2; Allen vẫn là 0.

In [34]:
ANSWER_175_6="""
SELECT p.personId, p.firstName, COUNT(a.addressId) AS addressCount
FROM Person AS p
LEFT JOIN Address AS a ON p.personId=a.personId
GROUP BY p.personId,p.firstName
ORDER BY p.personId;
"""
assert conn.execute(ANSWER_175_6).fetchall()==[(1,'Allen',0),(2,'Bob',1)]
show_sql(ANSWER_175_6)

Cột: ['personId', 'firstName', 'addressCount']
   (1, 'Allen', 0)
   (2, 'Bob', 1)
Số dòng: 2


[(1, 'Allen', 0), (2, 'Bob', 1)]

## 38. Template SQL cuối, pattern và checklist

**Chỉ nộp câu SQL này:**
```sql
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person AS p
LEFT JOIN Address AS a
    ON p.personId = a.personId;
```

**Pattern:** “Hãy lấy **tất cả A**, rồi **thêm thông tin B nếu có**” → suy nghĩ tới `A LEFT JOIN B`.

**Checklist trước khi submit:**
- [ ] Mình biết đâu là bảng **phải giữ tất cả dòng**? (Person)
- [ ] Phân biệt ID người với ID địa chỉ?
- [ ] Đã chọn đúng bốn cột, đúng tên và thứ tự?
- [ ] Hiểu `LEFT JOIN` đưa NULL khi không có khớp, không phải tự `INSERT NULL`?
- [ ] Không vô tình bỏ người chưa có địa chỉ do `INNER JOIN` hoặc `WHERE`?
- [ ] Biết kiểm tra `IS NULL`, không viết `= NULL`?
- [ ] Trên LeetCode chỉ nộp **SQL**; `sqlite3` chỉ để học?

**Tự hỏi thêm:** nếu người có hai địa chỉ thì sao? Nếu city trong một địa chỉ vốn NULL thì sao? Nếu Person rỗng thì sao? Nếu bạn trả lời đúng cả ba, nghĩa là đã hiểu bản chất LEFT JOIN.